In [2]:
import pickle
import numpy as np

# Load the best model (Logistic Regression)
with open('data/model_lr.pkl', 'rb') as f:
    model = pickle.load(f)

with open('data/scaler.pkl', 'rb') as f:
    sc = pickle.load(f)

with open('data/feature_names.pkl', 'rb') as f:
    feature_names = pickle.load(f)

print("✅ Model loaded and ready!")
print("   Model type:", type(model).__name__)
print("   Features  :", feature_names)

✅ Model loaded and ready!
   Model type: LogisticRegression
   Features  : ['Gender', 'Age', 'Occupation', 'Sleep Duration', 'Quality of Sleep', 'Physical Activity Level', 'Stress Level', 'BMI Category', 'Heart Rate', 'Daily Steps']


In [3]:
def lifestyle_score(sleep, activity, stress, bmi_cat, smoking):
    s = 0
    
    # Sleep: up to 25 points
    s += min(sleep / 8 * 25, 25)
    
    # Activity: up to 25 points
    s += min(activity / 90 * 25, 25)
    
    # Stress: up to 20 points (less stress = more points)
    s += max(0, (10 - stress) / 10 * 20)
    
    # BMI: up to 15 points
    bmi_pts = {'Normal': 15, 'Normal Weight': 15, 'Overweight': 8, 'Obese': 0}
    s += bmi_pts.get(bmi_cat, 10)
    
    # Non-smoker: 15 points
    s += 15 if not smoking else 0
    
    return round(s)

# Quick test
print("✅ Lifestyle score function created!")
print("Test — healthy person  :", lifestyle_score(8, 90, 2, 'Normal', False), "/100")
print("Test — unhealthy person:", lifestyle_score(5, 15, 9, 'Obese', True), "/100")

✅ Lifestyle score function created!
Test — healthy person  : 96 /100
Test — unhealthy person: 22 /100


In [4]:
def get_tips(risk, sleep, activity, stress, bmi_cat, smoking):
    tips = []
    
    if risk == 'High':
        tips.append("URGENT: Your risk is HIGH. Please consult a doctor soon.")
    
    if sleep < 6:
        tips.append("Sleep: Very low! Aim for at least 7-8 hours per night.")
    elif sleep < 7:
        tips.append("Sleep: Slightly low. Try to reach 7-9 hours per night.")
        
    if activity < 30:
        tips.append("Activity: Very low! Start with a 15 min walk every day.")
    elif activity < 60:
        tips.append("Activity: Good start! Try to reach 60 min/day.")
        
    if stress >= 8:
        tips.append("Stress: Very high! Try meditation or deep breathing exercises.")
    elif stress >= 6:
        tips.append("Stress: Moderate. Consider relaxation techniques daily.")
        
    if bmi_cat == 'Obese':
        tips.append("BMI: Consider consulting a nutritionist for a diet plan.")
    elif bmi_cat == 'Overweight':
        tips.append("BMI: Try to incorporate more vegetables and less processed food.")
        
    if smoking:
        tips.append("Smoking: Quitting smoking, it's the single best thing for your health.")
        
    if not tips:
        tips.append("✅ Excellent! You have a very healthy lifestyle. Keep it up!")
        
    return tips

print("Health tips function created!")
print("\nTest tips for unhealthy person:")
test_tips = get_tips('High', 5, 20, 9, 'Obese', True)
for t in test_tips:
     print(" ", t)

Health tips function created!

Test tips for unhealthy person:
  URGENT: Your risk is HIGH. Please consult a doctor soon.
  Sleep: Very low! Aim for at least 7-8 hours per night.
  Activity: Very low! Start with a 15 min walk every day.
  Stress: Very high! Try meditation or deep breathing exercises.
  BMI: Consider consulting a nutritionist for a diet plan.
  Smoking: Quitting smoking, it's the single best thing for your health.


In [6]:
import pandas as pd

def predict_health(age, sleep, activity, stress, bmi_cat, smoking,
                   gender=1, occupation=1, quality_sleep=7,
                   heart_rate=70, daily_steps=7000):
   
    bmi_map = {'Normal': 0, 'Normal Weight': 0, 'Obese': 1, 'Overweight': 2}
    bmi_num = bmi_map.get(bmi_cat, 0)
 
    user_input = pd.DataFrame([[gender, age, occupation, sleep, quality_sleep,
                                activity, stress, bmi_num, heart_rate, daily_steps]],
                               columns=feature_names)
   
    user_scaled = sc.transform(user_input)
    pred = model.predict(user_scaled)[0]
    risk_map = {0: 'High', 1: 'Low', 2: 'Medium'}
    risk = risk_map[pred]
  
    score = lifestyle_score(sleep, activity, stress, bmi_cat, smoking)
    tips  = get_tips(risk, sleep, activity, stress, bmi_cat, smoking)
    
    return risk, score, tips

print("✅ Prediction function updated!")

✅ Prediction function updated!


In [7]:
print("=" * 45)
print("TEST 1 — Unhealthy lifestyle person")
print("=" * 45)

risk, score, tips = predict_health(
    age=45,
    sleep=5.0,
    activity=15,
    stress=9,
    bmi_cat='Obese',
    smoking=True
)

print(f"  Age      : 45")
print(f"  Sleep    : 5.0 hrs/night")
print(f"  Activity : 15 min/day")
print(f"  Stress   : 9/10")
print(f"  BMI      : Obese")
print(f"  Smoking  : Yes")
print(f"\n  RISK LEVEL      : {risk}")
print(f"  LIFESTYLE SCORE : {score}/100")
print(f"\n  TIPS:")
for i, t in enumerate(tips, 1):
    print(f"    {i}. {t}")

TEST 1 — Unhealthy lifestyle person
  Age      : 45
  Sleep    : 5.0 hrs/night
  Activity : 15 min/day
  Stress   : 9/10
  BMI      : Obese
  Smoking  : Yes

  RISK LEVEL      : High
  LIFESTYLE SCORE : 22/100

  TIPS:
    1. URGENT: Your risk is HIGH. Please consult a doctor soon.
    2. Sleep: Very low! Aim for at least 7-8 hours per night.
    3. Activity: Very low! Start with a 15 min walk every day.
    4. Stress: Very high! Try meditation or deep breathing exercises.
    5. BMI: Consider consulting a nutritionist for a diet plan.
    6. Smoking: Quitting smoking, it's the single best thing for your health.


In [8]:
print("=" * 45)
print("TEST 2 — Healthy lifestyle person")
print("=" * 45)

risk, score, tips = predict_health(
    age=30,
    sleep=8.0,
    activity=75,
    stress=3,
    bmi_cat='Normal',
    smoking=False
)

print(f"  Age      : 30")
print(f"  Sleep    : 8.0 hrs/night")
print(f"  Activity : 75 min/day")
print(f"  Stress   : 3/10")
print(f"  BMI      : Normal")
print(f"  Smoking  : No")
print(f"\n  RISK LEVEL      : {risk}")
print(f"  LIFESTYLE SCORE : {score}/100")
print(f"\n  TIPS:")
for i, t in enumerate(tips, 1):
    print(f"    {i}. {t}")

TEST 2 — Healthy lifestyle person
  Age      : 30
  Sleep    : 8.0 hrs/night
  Activity : 75 min/day
  Stress   : 3/10
  BMI      : Normal
  Smoking  : No

  RISK LEVEL      : Low
  LIFESTYLE SCORE : 90/100

  TIPS:
    1. ✅ Excellent! You have a very healthy lifestyle. Keep it up!


In [9]:
print("=" * 45)
print("TEST 3 — Medium risk lifestyle person")
print("=" * 45)

risk, score, tips = predict_health(
    age=40,
    sleep=6.5,
    activity=40,
    stress=6,
    bmi_cat='Overweight',
    smoking=False
)

print(f"  Age      : 40")
print(f"  Sleep    : 6.5 hrs/night")
print(f"  Activity : 40 min/day")
print(f"  Stress   : 6/10")
print(f"  BMI      : Overweight")
print(f"  Smoking  : No")
print(f"\n  RISK LEVEL      : {risk}")
print(f"  LIFESTYLE SCORE : {score}/100")
print(f"\n  TIPS:")
for i, t in enumerate(tips, 1):
    print(f"    {i}. {t}")

TEST 3 — Medium risk lifestyle person
  Age      : 40
  Sleep    : 6.5 hrs/night
  Activity : 40 min/day
  Stress   : 6/10
  BMI      : Overweight
  Smoking  : No

  RISK LEVEL      : High
  LIFESTYLE SCORE : 62/100

  TIPS:
    1. URGENT: Your risk is HIGH. Please consult a doctor soon.
    2. Sleep: Slightly low. Try to reach 7-9 hours per night.
    3. Activity: Good start! Try to reach 60 min/day.
    4. Stress: Moderate. Consider relaxation techniques daily.
    5. BMI: Try to incorporate more vegetables and less processed food.


In [10]:
import ipywidgets as widgets
from IPython.display import display, clear_output

sleep_w    = widgets.FloatSlider(value=7.0, min=3.0, max=12.0, step=0.5,
                                  description='Sleep (hrs):',
                                  style={'description_width': '130px'})
activity_w = widgets.IntSlider(value=45, min=0, max=120,
                                description='Activity (min):',
                                style={'description_width': '130px'})
stress_w   = widgets.IntSlider(value=5, min=1, max=10,
                                description='Stress (1-10):',
                                style={'description_width': '130px'})
age_w      = widgets.IntSlider(value=30, min=18, max=80,
                                description='Age:',
                                style={'description_width': '130px'})
bmi_w      = widgets.Dropdown(options=['Normal', 'Overweight', 'Obese'],
                               description='BMI:',
                               style={'description_width': '130px'})
smoke_w    = widgets.Checkbox(value=False, description='Smoker?')
btn        = widgets.Button(description='Predict My Health Risk',
                             button_style='success',
                             layout=widgets.Layout(width='250px', height='40px'))
out        = widgets.Output()

def on_click(b):
    with out:
        clear_output()
        r, s, t = predict_health(
            age=age_w.value,
            sleep=sleep_w.value,
            activity=activity_w.value,
            stress=stress_w.value,
            bmi_cat=bmi_w.value,
            smoking=smoke_w.value
        )
        
        # Color based on risk
        color = {'Low': '32', 'Medium': '33', 'High': '31'}[r]
        
        print(f"\033[{color}m{'=' * 40}\033[0m")
        print(f"\033[{color}m  RISK LEVEL      : {r}\033[0m")
        print(f"\033[{color}m  LIFESTYLE SCORE : {s}/100\033[0m")
        print(f"\033[{color}m{'=' * 40}\033[0m")
        print("\nPersonalised Tips:")
        for i, tip in enumerate(t, 1):
            print(f"  {i}. {tip}")

btn.on_click(on_click)

print("Adjust the sliders and click the button!")
display(age_w, sleep_w, activity_w, stress_w, bmi_w, smoke_w, btn, out)

Adjust the sliders and click the button!


IntSlider(value=30, description='Age:', max=80, min=18, style=SliderStyle(description_width='130px'))

FloatSlider(value=7.0, description='Sleep (hrs):', max=12.0, min=3.0, step=0.5, style=SliderStyle(description_…

IntSlider(value=45, description='Activity (min):', max=120, style=SliderStyle(description_width='130px'))

IntSlider(value=5, description='Stress (1-10):', max=10, min=1, style=SliderStyle(description_width='130px'))

Dropdown(description='BMI:', options=('Normal', 'Overweight', 'Obese'), style=DescriptionStyle(description_wid…

Checkbox(value=False, description='Smoker?')

Button(button_style='success', description='Predict My Health Risk', layout=Layout(height='40px', width='250px…

Output()

In [11]:
def run_predictor():
    print("=" * 45)
    print("  HEALTHSENSE — Health Risk Predictor")
    print("=" * 45)
    
    age      = int(input("Enter age (18-80): "))
    sleep    = float(input("Enter sleep hours (3-12): "))
    activity = int(input("Enter activity minutes/day (0-120): "))
    stress   = int(input("Enter stress level (1-10): "))
    bmi_cat  = input("Enter BMI (Normal / Overweight / Obese): ")
    smoking  = input("Do you smoke? (yes/no): ").lower() == 'yes'
    
    r, s, t = predict_health(age, sleep, activity, stress, bmi_cat, smoking)
    
    print(f"\n{'=' * 45}")
    print(f"  RISK LEVEL      : {r}")
    print(f"  LIFESTYLE SCORE : {s}/100")
    print(f"{'=' * 45}")
    print("\nPersonalised Tips:")
    for i, tip in enumerate(t, 1):
        print(f"  {i}. {tip}")

run_predictor()

  HEALTHSENSE — Health Risk Predictor


Enter age (18-80):  18
Enter sleep hours (3-12):  7
Enter activity minutes/day (0-120):  20
Enter stress level (1-10):  6
Enter BMI (Normal / Overweight / Obese):  obese
Do you smoke? (yes/no):  no



  RISK LEVEL      : Medium
  LIFESTYLE SCORE : 60/100

Personalised Tips:
  1. Activity: Very low! Start with a 15 min walk every day.
  2. Stress: Moderate. Consider relaxation techniques daily.
